In [26]:
import os
import markdown
from dotenv import load_dotenv
from IPython.display import display, HTML, clear_output
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage

# Load API key securely
load_dotenv()

# Initialize the Gemini chat model
chat = ChatGoogleGenerativeAI(
    model="gemini-flash-lite-latest",
    temperature=0.1
)

In [27]:
# Helper function to render Markdown content from Right-to-Left (RTL)
# This improves the display quality of Persian/Arabic text in Jupyter
def print_md_rtl(content):
    html_content = markdown.markdown(content)
    display(HTML(f"""
        <div dir="rtl" style="text-align: right; font-family: Tahoma, Arial, sans-serif; line-height: 1.8; font-size: 16px;">
            {html_content}
        </div>
    """))

# Define the system instruction
system = SystemMessage(
    content="تو یک معلم زبان آلمانی هستی. پاسخ دانش‌آموز رو با نگاه آموزشی و با یک یا چند مثال رایج و عامیانه بده."
)

# Define multiple conversations for batch processing
messages = [
    [system, HumanMessage(content="شب به خیر به آلمانی چی میشه؟")],
    [system, HumanMessage(content="چطور به آلمانی بگم من در تهران متولد شدم؟")]
]

In [28]:
# Send multiple conversations to the model at once (Batch Process)
print("⏳ Processing batch requests...\n")
responses = chat.batch(messages)

# Iterate through the responses and display them using the RTL markdown helper
for idx, response in enumerate(responses):
    print(f"--- Conversation {idx + 1} ---")
    
    # Gemini sometimes returns a list of blocks (multimodal format) instead of a plain string.
    # We must safely extract the text to prevent 'AttributeError' during Markdown rendering.
    if isinstance(response.content, list):
        safe_text = response.content[0].get("text", "")
    else:
        safe_text = response.content
        
    # Render the extracted plain text
    print_md_rtl(safe_text)
    print("*" * 70)

⏳ Processing batch requests...

--- Conversation 1 ---


**********************************************************************
--- Conversation 2 ---


**********************************************************************


In [29]:
# ==========================================
# Streaming Output (Live Typing Effect)
# ==========================================

print("⏳ Streaming response for the first conversation...\n")

full_response = ""

# Stream the response token-by-token
for chunk in chat.stream(messages[0]):
    
    # Safely extract text in case the chunk is returned as a list
    if isinstance(chunk.content, list):
        # بررسی می‌کنیم که لیست خالی نباشد (برای جلوگیری از IndexError در آخرین تکه)
        chunk_text = chunk.content[0].get("text", "") if len(chunk.content) > 0 else ""
    else:
        chunk_text = chunk.content
        
    full_response += chunk_text
    
    # Clear the previous output before printing the updated string
    clear_output(wait=True)
    print_md_rtl(full_response)

print("\n" + "*" * 70 + "\n")

# ==========================================
# Streaming the second conversation
# ==========================================

print("⏳ Streaming response for the second conversation...\n")

full_response_2 = ""

for chunk in chat.stream(messages[1]):
    
    # Safely extract text
    if isinstance(chunk.content, list):
        chunk_text = chunk.content[0].get("text", "") if len(chunk.content) > 0 else ""
    else:
        chunk_text = chunk.content
        
    full_response_2 += chunk_text
    
    clear_output(wait=True)
    print_md_rtl(full_response_2)